In [59]:
# импортируем библиотеки
import pandas as pd
from datetime import datetime, date

# словари макрорегионов и федеральных округов + утилиты нормализации
from macroregions import COUNTRY_MACRO_REGIONS
from russian_fo import RUSSIAN_REGIONS
from required_fields import rules

In [60]:
events_name = f"events {datetime.now().strftime('%Y-%m-%d %H-%M')}"
essay_name = f"essay {datetime.now().strftime('%Y-%m-%d %H-%M')}"
video_name = f"video {datetime.now().strftime('%Y-%m-%d %H-%M')}"

print(events_name)
print(essay_name)
print(video_name)

events 2026-03-13 10-39
essay 2026-03-13 10-39
video 2026-03-13 10-39


In [87]:
try:
    df = pd.read_csv(f'C:/Users/D.Loskutnikov/Downloads/{events_name}.csv', sep=';')
except:
    df = pd.read_csv(f'/Users/bis.partnermail.ru/Downloads/{events_name}.csv', sep=';')

/var/folders/bw/9z2dh6pd1w10bv3kg9h4mr0m0000gn/T/ipykernel_91664/304363769.py:4: DtypeWarning: Columns (0: Статус ДВФМ, 1: Комментарий менеджера по конкурсн, 2: Регион РП 2024 #visitreg2024, 3: Маршрут блог-тура 80 лет ВОВ #routethematicblog, 4: Регион РП ШУМ 2024 #whichcity4, 5: Кем выдан документ #issuingauthorityadd, 6: Поделись своими достижениями и опытом реализации проектов в выбранных направлениях деятельности #sharexperience, 7: Наименование учебного учреждения (на английском) #educationalnameadd, 8: Наименование учебного учреждения (на английском) #educationalnameadd2, 9: Специальность (на английском) #studytypeadd2, 10: Название языка #languageName4, 11: Название языка #languageName5, 12: Место работы: название организации (на английском) #placeofemploymentadd, 13: Должность (на английском) #jobtitleadd, 14: Место работы: название организации (на русском) #placeofemploymentadd2, 15: Место работы: название организации (на английском) #placeofemploymentadd2, 16: Должность (на р

In [88]:
col = [
    'ID',
    'Email #email',
    'Гражданство',
    'Страна проживания',
    'Регион проживания',
    'Возраст',
    'Состояние',
    'Готов добраться за свой счёт #travelownexpense',
    'Участие в РП (да/нет) #expeditionprogram',
    'Дата создания заявки',
    'Имя на русском #firstName.ru',
    'Имя на английском #firstName.en',
    'Нет имени',
    'Фамилия на русском #lastName.ru',
    'Фамилия на английском #lastName.en',
    'Нет фамилии',
    'Отчество на русском #patronymic.ru',
    'Отчество на английском #patronymic.en',
    'Нет отчества',
    'Пол #sex',
    'Дата рождения',
    'Адрес фактического места проживания #addressresidence',
    'Родной язык #nativelanguage',
    'Род деятельность #statusactivity',
    'Направление деятельности #fieldwork',
    'Номер телефона #phone',
    'Фотография #photo',
    'Город фактического места проживания #cityresidence',
    'Язык переписки #correspondencelanguage',
    'Тип питания #dietype',
    'Уровень владения английским языком #englevel',
    'Уровень владения русским языком #ruslevel'
    ]

In [89]:
df = df[col].copy()

In [90]:
try:
    df_video = pd.read_csv(f'/Users/bis.partnermail.ru/Downloads/{video_name}.csv', sep=';')
    df_essay = pd.read_csv(f'/Users/bis.partnermail.ru/Downloads/{essay_name}.csv', sep=';')
except:
    df_video = pd.read_csv(f'C:/Users/D.Loskutnikov/Downloads/{video_name}.csv', sep=';')
    df_essay = pd.read_csv(f'C:/Users/D.Loskutnikov/Downloads/{essay_name}.csv', sep=';')

In [91]:
# Фильтрация email с @wyffest.com
df = df[~df['Email #email'].str.contains('@wyffest.com', case=False, na=False)]
# Убираем отозванные заявки
df = df[(df['Состояние'] == "Черновик") | (df['Состояние'] == "На рассмотрении")]

In [92]:
# Подготовка данных для основной статистики
df['is_child'] = df['Возраст'].between(14, 17)
df['is_rf_citizenship'] = df['Гражданство'] == 'Россия'
df['is_rf_residence'] = df['Страна проживания'] == 'Россия'

# Фильтрация эссе и видео по статусу "Отправлена"
df_essay_filtered = df_essay[df_essay['Состояние'] == 'Отправлена'].copy()
df_video_filtered = df_video[df_video['Состояние'] == 'Отправлена'].copy()

# Убираем @wyffest.com из эссе и видео
df_essay_filtered = df_essay_filtered[~df_essay_filtered['Автор'].str.contains('@wyffest.com', case=False, na=False)]
df_video_filtered = df_video_filtered[~df_video_filtered['Автор'].str.contains('@wyffest.com', case=False, na=False)]

# Оставляем только уникальных авторов
df_essay_unique = df_essay_filtered.drop_duplicates(subset=['Автор'])
df_video_unique = df_video_filtered.drop_duplicates(subset=['Автор'])

In [93]:
# # Подтягиваем возраст и категории из основной df
# df_essay_merged = df.merge(
#     df_essay_unique[['Автор', 'Состояние']],
#     left_on='Email #email',
#     right_on='Автор',
#     how='left'
# )

# df_essay_merged = df_essay_merged.rename(columns={
#     'Состояние_y':'Статус эссе'
# })

# df_essay_merged = df_essay_merged.drop(columns={"Автор"})

In [94]:
# Присоединяем к уникальной заявке статус эссе и статус видеовизитки.
# Если эссе или видео нет, то присоединится NaN.
df_merged = df.drop_duplicates(subset=['Email #email']).merge(
    df_essay_unique[['Автор', 'Состояние']].rename(columns={'Автор': 'Email #email', 'Состояние': 'Статус эссе'}),
    on='Email #email',
    how='left'
).merge(
    df_video_unique[['Автор', 'Состояние']].rename(columns={'Автор': 'Email #email', 'Состояние': 'Статус видеовизитки'}),
    on='Email #email',
    how='left'
)

# Статус заявки нам нужен из основной df (там это "Состояние"), поэтому добавим явно:
df_merged['Статус заявки'] = df_merged['Состояние']
df_merged = df_merged.drop(columns=['Состояние'])

In [95]:
# Общая статистика
total = len(df_merged)
children_14_17 = df_merged['is_child'].sum()
adults_18_35 = total - children_14_17

# Общая статистика по эссе и видео
total_essay = len(df_merged[df_merged['Статус эссе'].notna()])
essay_children = df_merged[df_merged['Статус эссе'].notna() & df_merged['is_child']].shape[0]
essay_adults = total_essay - essay_children

total_video = len(df_merged[df_merged['Статус видеовизитки'].notna()])
video_children = df_merged[df_merged['Статус видеовизитки'].notna() & df_merged['is_child']].shape[0]
video_adults = total_video - video_children

In [96]:
# РФ
rf = df_merged[(df_merged['is_rf_citizenship']) & (df_merged['is_rf_residence'])]
rf_total = len(rf)
rf_children = rf['is_child'].sum()
rf_essay = len(rf[rf['Статус эссе'].notna()])
rf_video = len(rf[rf['Статус видеовизитки'].notna()])
rf_children_essay = len(rf[(rf['Статус эссе'].notna()) & (rf['is_child'])])
rf_children_video = len(rf[(rf['Статус видеовизитки'].notna()) & (rf['is_child'])])

In [97]:
# ИН
foreign = df_merged[~df_merged['is_rf_citizenship'] & ~df_merged['is_rf_residence']]
foreign_total = len(foreign)
foreign_children = foreign['is_child'].sum()
foreign_essay = len(foreign[foreign['Статус эссе'].notna()])
foreign_video = len(foreign[foreign['Статус видеовизитки'].notna()])
foreign_children_essay = len(foreign[(foreign['Статус эссе'].notna()) & (foreign['is_child'])])
foreign_children_video = len(foreign[(foreign['Статус видеовизитки'].notna()) & (foreign['is_child'])])

In [98]:
# ИН в РФ
foreign_in_rf = df_merged[~df_merged['is_rf_citizenship'] & df_merged['is_rf_residence']]
foreign_in_rf_total = len(foreign_in_rf)
foreign_in_rf_children = foreign_in_rf['is_child'].sum()
foreign_in_rf_essay = len(foreign_in_rf[foreign_in_rf['Статус эссе'].notna()])
foreign_in_rf_video = len(foreign_in_rf[foreign_in_rf['Статус видеовизитки'].notna()])
foreign_in_rf_children_essay = len(foreign_in_rf[(foreign_in_rf['Статус эссе'].notna()) & (foreign_in_rf['is_child'])])
foreign_in_rf_children_video = len(foreign_in_rf[(foreign_in_rf['Статус видеовизитки'].notna()) & (foreign_in_rf['is_child'])])

In [99]:
# Соотечественники
compatriots = df_merged[df_merged['is_rf_citizenship'] & ~df_merged['is_rf_residence']]
compatriots_total = len(compatriots)
compatriots_children = compatriots['is_child'].sum()
compatriots_essay = len(compatriots[compatriots['Статус эссе'].notna()])
compatriots_video = len(compatriots[compatriots['Статус видеовизитки'].notna()])
compatriots_children_essay = len(compatriots[(compatriots['Статус эссе'].notna()) & (compatriots['is_child'])])
compatriots_children_video = len(compatriots[(compatriots['Статус видеовизитки'].notna()) & (compatriots['is_child'])])

In [100]:
# Количество стран
countries_count = df_merged['Страна проживания'].nunique()

In [101]:
print(f"""
**ЗАЯВОЧНАЯ КАМПАНИЯ | {datetime.now().strftime('%d.%m.%Y, %H:%M')}**

**Всего** — {total} заявок, из них:
14-17 лет — {children_14_17} чел.,
18-35 лет — {adults_18_35} чел.

**Отправлено эссе** — {total_essay}
14-17 лет — {essay_children} эссе,
18-35 лет — {essay_adults} эссе

**Отправлено видео** — {total_video}
14-17 лет — {video_children} видео,
18-35 лет — {video_adults} видео

🇷🇺 **РФ** — {rf_total} чел.,
**Отправлено эссе** — {rf_essay},
**Отправлено видео** — {rf_video}
🔵 **из них дети (14-17)** — {rf_children} чел.
**Отправлено эссе** — {rf_children_essay},
**Отправлено видео** — {rf_children_video}

🇷🇺 **ИН** — {foreign_total} чел.,
**Отправлено эссе** — {foreign_essay},
**Отправлено видео** — {foreign_video}
🔵 **из них дети (14-17)** — {foreign_children} чел.
**Отправлено эссе** — {foreign_children_essay},
**Отправлено видео** — {foreign_children_video}

🇷🇺 **ИН в РФ** — {foreign_in_rf_total} чел.,
**Отправлено эссе** — {foreign_in_rf_essay},
**Отправлено видео** — {foreign_in_rf_video}
🔵 **из них дети (14-17)** — {foreign_in_rf_children} чел.
**Отправлено эссе** — {foreign_in_rf_children_essay},
**Отправлено видео** — {foreign_in_rf_children_video}

🇷🇺 **Соотечественники** — {compatriots_total} чел.,
**Отправлено эссе** — {compatriots_essay},
**Отправлено видео** — {compatriots_video}
🔵 **из них дети (14-17)** — {compatriots_children} чел.
**Отправлено эссе** — {compatriots_children_essay},
**Отправлено видео** — {compatriots_children_video}

🌍 **Количество стран** — {countries_count}
""")


**ЗАЯВОЧНАЯ КАМПАНИЯ | 13.03.2026, 10:40**

**Всего** — 31783 заявок, из них:
14-17 лет — 4855 чел.,
18-35 лет — 26928 чел.

**Отправлено эссе** — 4810
14-17 лет — 663 эссе,
18-35 лет — 4147 эссе

**Отправлено видео** — 910
14-17 лет — 130 видео,
18-35 лет — 780 видео

🇷🇺 **РФ** — 14289 чел.,
**Отправлено эссе** — 1928,
**Отправлено видео** — 397
🔵 **из них дети (14-17)** — 3737 чел.
**Отправлено эссе** — 442,
**Отправлено видео** — 68

🇷🇺 **ИН** — 15914 чел.,
**Отправлено эссе** — 2569,
**Отправлено видео** — 432
🔵 **из них дети (14-17)** — 1066 чел.
**Отправлено эссе** — 207,
**Отправлено видео** — 57

🇷🇺 **ИН в РФ** — 1436 чел.,
**Отправлено эссе** — 288,
**Отправлено видео** — 73
🔵 **из них дети (14-17)** — 10 чел.
**Отправлено эссе** — 1,
**Отправлено видео** — 0

🇷🇺 **Соотечественники** — 144 чел.,
**Отправлено эссе** — 25,
**Отправлено видео** — 8
🔵 **из них дети (14-17)** — 42 чел.
**Отправлено эссе** — 13,
**Отправлено видео** — 5

🌍 **Количество стран** — 171



In [102]:
df_merged['Макрорегион'] = df_merged['Страна проживания'].map(COUNTRY_MACRO_REGIONS).fillna('Нет макро')
df_merged['Федеральный округ'] = df_merged['Регион проживания'].map(RUSSIAN_REGIONS).fillna('Нет ФО')

In [103]:
def category(age):
    if 14 <= age <= 17:
        return '14-17'
    elif 18 <= age <= 35:
        return '18-35'
    else:
        return 'Другая категория'  # или 'Не подходит'

df_merged['Категория участника'] = df_merged['Возраст'].apply(category)

In [104]:
df_merged['Дата создания заявки'] = pd.to_datetime(df_merged['Дата создания заявки'], errors='coerce').dt.strftime("%d.%m.%Y")

In [105]:
def is_profile_complete(row):
    # Список обязательных полей из required_fields.py (3-22)
    required_fields = [
        'ID',
        'Email #email',
        'Гражданство',
        'Страна проживания',
        'Пол #sex',
        'Дата рождения',
        'Дата рождения',
        'Номер телефона #phone',
        'Фотография #photo',
        'Город фактического места проживания #cityresidence',
        'Адрес фактического места проживания #addressresidence',
        'Родной язык #nativelanguage',
        'Язык переписки #correspondencelanguage',
        'Тип питания #dietype',
        'Род деятельность #statusactivity',
        'Направление деятельности #fieldwork',
        'Уровень владения английским языком #englevel',
        'Уровень владения русским языком #ruslevel'
    ]
    # Группы "или-или" для ФИО
    either_or_groups = [
        (["Имя на русском #firstName.ru", "Имя на английском #firstName.en", "Нет имени"], "Нет имени"),
        (["Фамилия на русском #lastName.ru", "Фамилия на английском #lastName.en", "Нет фамилии"], "Нет фамилии"),
        (["Отчество на русском #patronymic.ru", "Отчество на английском #patronymic.en", "Нет отчества"], "Нет отчества"),
    ]
    # Проверяем обязательные поля
    for field in required_fields:
        if pd.isna(row.get(field)) or str(row.get(field)).strip() == '':
            return False
    # Проверяем либо хотя бы одно из имен (на русском/английском), либо "Нет имени" == "Да"
    for fields, neg_field in either_or_groups:
        # индексы 0,1 - это имя/фамилия/отчество, 2 - "Нет имя" и т.д.
        if (
            (pd.isna(row.get(fields[0])) or str(row.get(fields[0])).strip() == '') and
            (pd.isna(row.get(fields[1])) or str(row.get(fields[1])).strip() == '')
        ):
            # Если оба пустые, то проверяем "Нет ..."
            if str(row.get(fields[2])).strip() != 'Да':
                return False
    return True

df_merged['Профиль заполнен полностью'] = df_merged.apply(is_profile_complete, axis=1)

In [106]:
# import random

# required_fields = [
#         'ID',
#         'Email #email',
#         'Гражданство',
#         'Страна проживания',
#         'Регион проживания',
#         'Пол #sex',
#         'Дата рождения',
#         'Дата рождения',
#         'Номер телефона #phone',
#         'Фотография #photo',
#         'Город фактического места проживания #cityresidence',
#         'Адрес фактического места проживания #addressresidence',
#         'Родной язык #nativelanguage',
#         'Язык переписки #correspondencelanguage',
#         'Тип питания #dietype',
#         'Род деятельность #statusactivity',
#         'Учусь в России #studyrussia',
#         'Направление деятельности #fieldwork',
#         'Уровень владения английским языком #englevel',
#         'Уровень владения русским языком #ruslevel'
#     ]
#     # Группы "или-или" для ФИО
# either_or_groups = [
#         (["Имя на русском #firstName.ru", "Имя на английском #firstName.en", "Нет имени"], "Нет имени"),
#         (["Фамилия на русском #lastName.ru", "Фамилия на английском #lastName.en", "Нет фамилии"], "Нет фамилии"),
#         (["Отчество на русском #patronymic.ru", "Отчество на английском #patronymic.en", "Нет отчества"], "Нет отчества"),
#     ]

# # Выберем 5 случайных индексов
# random_indices = random.sample(list(df_merged.index), 5)

# for idx in random_indices:
#     row = df_merged.loc[idx]
#     missing_fields = []
#     # Проверяем обязательные поля
#     for field in required_fields:
#         if pd.isna(row.get(field)) or str(row.get(field)).strip() == '':
#             missing_fields.append(field)
#     # Проверяем "или-или" группы для ФИО
#     for fields, neg_field in either_or_groups:
#         if (
#             (pd.isna(row.get(fields[0])) or str(row.get(fields[0])).strip() == '') and
#             (pd.isna(row.get(fields[1])) or str(row.get(fields[1])).strip() == '')
#         ):
#             if str(row.get(fields[2])).strip() != 'Да':
#                 # Добавляем недостающие поля: оба имени и нет "Да" в "Нет имя"
#                 missing_fields.append(f"{fields[0]} или {fields[1]} или '{fields[2]}' == 'Да'")
#     print(f"\nСтрока c индексом {idx}:")
#     print(row)
#     if missing_fields:
#         print("Недостающие поля для полного профиля:")
#         for fld in missing_fields:
#             print(f"  - {fld}")
#     else:
#         print("Все необходимые поля заполнены, профиль считается полным.")

In [107]:
# pd.set_option('display.max_columns', None)
# display(df_merged)

In [108]:
# # Получим список пользователей с неполным профилем и статусом "На рассмотрении"
# incomplete_review = df_merged[(df_merged['Профиль заполнен полностью'] == False) & (df_merged['Статус заявки'] == "Черновик")]

# # Список обязательных полей
# from required_fields import rules
# required_fields = rules['required']

# # Поля "или-или" для ФИО
# either_or_groups = rules['either_or']

# def get_missing_fields(row):
#     missing = []
#     # Обычные обязательные поля
#     for field in required_fields:
#         if pd.isna(row.get(field)) or str(row.get(field)).strip() == '':
#             missing.append(field)
#     # Проверка "или-или"-групп
#     for fields in either_or_groups:
#         # например: ['Имя на русском', 'Имя на английском', 'Нет имени']
#         field1, field2, none_field = fields
#         f1_empty = pd.isna(row.get(field1)) or str(row.get(field1)).strip() == ''
#         f2_empty = pd.isna(row.get(field2)) or str(row.get(field2)).strip() == ''
#         none_yes = (str(row.get(none_field)).strip() == 'Да')
#         if f1_empty and f2_empty and not none_yes:
#             # Нужно заполнить хотя бы один, если нет "Нет <field>" == "Да"
#             missing.append(f"{field1} или {field2} или '{none_field}'=='Да'")
#     return missing

# # Для каждой строки покажем недостающие поля
# for idx, row in incomplete_review.iterrows():
#     missing = get_missing_fields(row)
#     print(f"Строка {idx} (ID: {row.get('ID')}):")
#     if missing:
#         print("  Не заполнены поля:")




#         for m in missing:
#             print(f"   - {m}")
#     else:
#         print("  Причина неясна — все обязательные поля заполнены.")

In [109]:
df_merged[(df_merged['Страна проживания'] == "Венесуэла") & (df_merged['Возраст'].between(14,17))]

,ID,Email #email,Гражданство,Страна проживания,Регион проживания,Возраст,Готов добраться за свой счёт #travelownexpense,Участие в РП (да/нет) #expeditionprogram,Дата создания заявки,Имя на русском #firstName.ru,...,is_child,is_rf_citizenship,is_rf_residence,Статус эссе,Статус видеовизитки,Статус заявки,Макрорегион,Федеральный округ,Категория участника,Профиль заполнен полностью
1480,d39499c2-c5fd-4130-88f4-0a0f41fe1043,joharojas201210@gmail.com,Венесуэла,Венесуэла,NaN,15.0,NaN,NaN,05.02.2026,Джоанна,...,True,False,False,Отправлена,NaN,На рассмотрении,Америка,Нет ФО,14-17,True
13362,2a3f291a-21ca-45e1-bfac-a13a4fea1794,nanjeligarciagallardo@gmail.com,Венесуэла,Венесуэла,NaN,16.0,NaN,NaN,14.02.2026,Нанйели,...,True,False,False,Отправлена,NaN,На рассмотрении,Америка,Нет ФО,14-17,True


In [110]:
df_merged['Профиль заполнен полностью'].value_counts()

Профиль заполнен полностью
True     23890
False     7893
Name: count, dtype: int64

In [111]:
df_merged['Статус заявки'].value_counts()

Статус заявки
На рассмотрении    20740
Черновик           11043
Name: count, dtype: int64

In [113]:
df_merged[[
    'ID',
    'Гражданство',
    'Страна проживания',
    'Регион проживания',
    'Возраст',
    'Статус заявки',
    'is_child',
    'is_rf_citizenship',
    'is_rf_residence',
    'Готов добраться за свой счёт #travelownexpense',
    'Участие в РП (да/нет) #expeditionprogram',
    'Макрорегион',
    'Федеральный округ',
    'Статус эссе',
    'Статус видеовизитки',
    'Категория участника',
    'Дата создания заявки',
    'Профиль заполнен полностью'
]].to_excel(
    r"data/Выгрузка данных от {0}.xlsx".format(date.today()),
    index=False
)